##### Modulated VBM

In [5]:
#import statements
import pandas as pd
import matplotlib.pyplot as plt
import subprocess
import numpy as np
import nilearn.image
import nilearn.plotting
import tempfile
import subprocess, shutil
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import copy
import shutil
from torch.utils.data import random_split
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from pathlib import Path
import ants
import pydicom
import nibabel as nib
import os
from glob import glob
from tqdm import tqdm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torchinfo import summary
import nilearn

os.environ.setdefault("FSLDIR", "/Users/william.wakefield/fsl")
fsl_bin = f"{os.environ['FSLDIR']}/share/fsl/bin"
if fsl_bin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = fsl_bin + os.pathsep + os.environ.get("PATH", "")
os.environ.setdefault("FSLOUTPUTTYPE", "NIFTI_GZ")

'NIFTI_GZ'

In [ ]:
vbm_root = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm")
raw_dir = vbm_root / "t1_long_raw"
strip_dir = vbm_root / "t1_long_skull_strip"

raw_files = sorted(raw_dir.glob("*.nii.gz"))
print(f"Found {len(raw_files)} raw native T1 images to skull-strip")

Found 802 raw native T1 images to skull-strip


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

raw_files = sorted(raw_dir.glob("*.nii.gz"))
print(f"Skull-strip {len(raw_files)} T1 volumes (bet -R -f 0.40)")


def _run_bet(raw_path, out_nii):
    result = subprocess.run(
        ["bet", str(raw_path), str(out_nii), "-R", "-f", "0.40"],
        capture_output=True, text=True,
    )
    return raw_path.stem, result.returncode, result.stderr.strip()

failed = []
skipped = 0
tasks = []

for raw_path in raw_files:
    out_nii = strip_dir / f"{raw_path.stem}.nii.gz"
    if out_nii.exists():
        skipped += 1
        continue
    tasks.append((raw_path, out_nii))

bet_workers = int(os.environ["BET_MAX_WORKERS"]) if os.environ.get("BET_MAX_WORKERS") else (os.cpu_count() or 8)
max_workers = max(1, min(len(tasks) or 1, bet_workers))
print(f"BET: {len(tasks)} jobs, {max_workers} workers ({skipped} already done)")

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(_run_bet, raw, out) for raw, out in tasks]
    for fut in tqdm(as_completed(futures), total=len(tasks), desc="BET native T1"):
        stem, rc, err = fut.result()
        if rc != 0:
            failed.append((stem, err))

stripped = list(strip_dir.glob("*.nii.gz"))
print(f"\nNative skull stripping complete: {len(stripped)} images")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Skull-strip 802 T1 volumes (bet -R -f 0.40)
BET: 802 jobs, 16 workers (0 already done)


BET native T1:  27%|██▋       | 220/802 [02:41<07:07,  1.36it/s]


In [4]:
vbm_root = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm")
strip_dir = vbm_root / "t1_long_skull_strip"
seg_dir = vbm_root / "t1_long_seg_native"

stripped_files = sorted(strip_dir.glob("*.nii.gz"))
print(f"Found {len(stripped_files)} native skull-stripped T1 images to segment")

Found 802 native skull-stripped T1 images to segment


In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def _run_fast(strip_path, out_base):
    result = subprocess.run(
        [
            "fast",
            "-t", "1",
            "-n", "3",
            "-B",
            "-b",
            "-o", str(out_base),
            str(strip_path),
        ],
        capture_output=True, text=True,
    )
    stem = strip_path.name.replace(".nii.gz", "")
    return stem, result.returncode, result.stderr.strip()

failed = []
skipped = 0
tasks = []

for strip_path in stripped_files:
    stem = strip_path.name.replace(".nii.gz", "")
    subject_dir = seg_dir / stem
    subject_dir.mkdir(parents=True, exist_ok=True)
    out_base = subject_dir / stem

    pve_csf = subject_dir / f"{stem}_pve_0.nii.gz"
    pve_gm = subject_dir / f"{stem}_pve_1.nii.gz"
    pve_wm = subject_dir / f"{stem}_pve_2.nii.gz"

    if pve_csf.exists() and pve_gm.exists() and pve_wm.exists():
        skipped += 1
        continue

    tasks.append((strip_path, out_base))

max_workers = max(1, (os.cpu_count() or 4) - 2)
print(f"Running {len(tasks)} FASTs in parallel with {max_workers} workers "
      f"({skipped} skipped, already existed)")

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(_run_fast, sp, ob) for sp, ob in tasks]
    for fut in tqdm(as_completed(futures), total=len(tasks), desc="FAST native T1"):
        stem, rc, err = fut.result()
        if rc != 0:
            failed.append((stem, err))

csf_maps = list(seg_dir.glob("*/*_pve_0.nii.gz"))
gm_maps = list(seg_dir.glob("*/*_pve_1.nii.gz"))
wm_maps = list(seg_dir.glob("*/*_pve_2.nii.gz"))
print(f"\nNative FAST segmentation complete:")
print(f"  {len(csf_maps)} CSF maps (pve_0)")
print(f"  {len(gm_maps)} GM maps (pve_1)")
print(f"  {len(wm_maps)} WM maps (pve_2)")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Running 802 FASTs in parallel with 14 workers (0 skipped, already existed)


FAST native T1: 100%|██████████| 802/802 [4:52:35<00:00, 21.89s/it]  


Native FAST segmentation complete:
  802 CSF maps (pve_0)
  802 GM maps (pve_1)
  802 WM maps (pve_2)
  (0 skipped, already existed)


In [6]:
# ── BET: skull-strip mean b=0 for each DTI subject ───────────────────────────
import nibabel as nib, numpy as np, subprocess, os, tempfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

dti_raw_dir   = Path("model_data/adni/dti_long_data/dti_long_raw")
dti_bet_dir   = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm/dti_long_bet")
dti_bet_dir.mkdir(parents=True, exist_ok=True)

def _extract_mean_b0(nii_path: Path, bval_path: Path) -> np.ndarray | None:
    bvals = np.loadtxt(str(bval_path)).flatten()
    b0_idx = np.where(bvals < 50)[0]
    if len(b0_idx) == 0:
        return None
    img = nib.load(str(nii_path))
    data = img.get_fdata(dtype=np.float32)
    return data[..., b0_idx].mean(axis=3), img.affine, img.header

def _bet_b0(stem: str) -> tuple[str, str]:
    nii_path  = dti_raw_dir / f"{stem}.nii.gz"
    bval_path = dti_raw_dir / f"{stem}.bval"
    out_b0    = dti_bet_dir / f"{stem}_meanb0.nii.gz"
    out_mask  = dti_bet_dir / f"{stem}_mask.nii.gz"

    if out_b0.exists() and out_mask.exists():
        return stem, "skip"
    if not nii_path.exists() or not bval_path.exists():
        return stem, "fail:missing_raw"

    try:
        result = _extract_mean_b0(nii_path, bval_path)
        if result is None:
            return stem, "fail:no_b0"
        mean_b0, affine, header = result

        # write mean b0 to temp, run BET
        with tempfile.NamedTemporaryFile(suffix=".nii.gz", delete=False) as f:
            tmp_b0 = f.name
        nib.save(nib.Nifti1Image(mean_b0, affine, header), tmp_b0)
        nib.save(nib.as_closest_canonical(nib.load(tmp_b0)), str(out_b0))
        os.unlink(tmp_b0)

        bet_prefix = dti_bet_dir / f"{stem}_bet"
        r = subprocess.run(
            ["bet", str(out_b0), str(bet_prefix), "-m", "-n", "-f", "0.25"],
            capture_output=True, text=True,
        )
        # bet writes <prefix>_mask.nii.gz; rename to our convention
        bet_mask_path = dti_bet_dir / f"{stem}_bet_mask.nii.gz"
        if bet_mask_path.exists():
            bet_mask_path.rename(out_mask)
        if r.returncode != 0:
            return stem, f"fail:bet_rc{r.returncode}"
        return stem, "ok"
    except Exception as e:
        return stem, f"fail:{e}"

# collect stems from raw dir
all_bval_files = sorted(dti_raw_dir.glob("*.bval"))
dti_stems = [p.stem for p in all_bval_files]   # stem = no extension

skipped, failed = 0, []
tasks = [s for s in dti_stems
         if not (dti_bet_dir / f"{s}_meanb0.nii.gz").exists()
         or not (dti_bet_dir / f"{s}_mask.nii.gz").exists()]
skipped = len(dti_stems) - len(tasks)

max_workers = max(1, (os.cpu_count() or 4) - 2)
print(f"BET: {len(tasks)} to run, {skipped} skipped, {max_workers} workers")

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {ex.submit(_bet_b0, s): s for s in tasks}
    for fut in tqdm(as_completed(futures), total=len(tasks), desc="BET b0"):
        stem, status = fut.result()
        if "fail" in status:
            failed.append((stem, status))

masks = list(dti_bet_dir.glob("*_mask.nii.gz"))
print(f"\nBET complete: {len(masks)} masks | {len(failed)} failed")
if failed:
    for s, e in failed[:10]: print(f"  {s}: {e}")

BET: 803 to run, 0 skipped, 14 workers


BET b0: 100%|██████████| 803/803 [02:02<00:00,  6.54it/s]


BET complete: 802 masks | 1 failed
  I1612816_128_S_0205a: fail:no_b0


In [7]:
# ── dtifit ────────────────────────────────────────────────────────────────────
dtifit_dir = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm/dti_long_dtifit")
dtifit_dir.mkdir(parents=True, exist_ok=True)

EXCLUDE_STEMS = {"141_S_6116"}   # <6 DWI directions — tensor underdetermined

def check_bval_adequacy(bval_path: Path) -> tuple[bool, str]:
    bvals = np.loadtxt(str(bval_path)).flatten()
    n_b0  = int(np.sum(bvals < 50))
    n_dwi = int(np.sum(bvals >= 500))
    if n_b0 == 0:   return False, "no b=0 volumes"
    if n_dwi < 6:   return False, f"only {n_dwi} DWI directions — need ≥6"
    return True, f"ok ({n_b0} b=0, {n_dwi} DWI)"

def _run_dtifit(stem: str) -> tuple[str, str]:
    if stem in EXCLUDE_STEMS:
        return stem, "skip:excluded"

    nii_path  = dti_raw_dir  / f"{stem}.nii.gz"
    bval_path = dti_raw_dir  / f"{stem}.bval"
    bvec_path = dti_raw_dir  / f"{stem}.bvec"
    mask_path = dti_bet_dir  / f"{stem}_mask.nii.gz"
    out_dir   = dtifit_dir   / stem
    out_md    = out_dir      / f"{stem}_MD.nii.gz"
    out_fa    = out_dir      / f"{stem}_FA.nii.gz"

    if out_md.exists() and out_fa.exists():
        return stem, "skip"
    for p in [nii_path, bval_path, bvec_path, mask_path]:
        if not p.exists():
            return stem, f"fail:missing_{p.name}"

    ok, msg = check_bval_adequacy(bval_path)
    if not ok:
        return stem, f"fail:bval_{msg}"

    out_dir.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(
        [
            "dtifit",
            "-k", str(nii_path),
            "-o", str(out_dir / stem),
            "-m", str(mask_path),
            "-r", str(bvec_path),
            "-b", str(bval_path),
            "--sse",          # save sum-of-squared errors (useful QC map)
        ],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        return stem, f"fail:dtifit_rc{r.returncode} {r.stderr[:200]}"
    return stem, "ok"

tasks = [s for s in dti_stems
         if not (dtifit_dir / s / f"{s}_MD.nii.gz").exists()
         and s not in EXCLUDE_STEMS]
skipped_n = len(dti_stems) - len(tasks)

print(f"dtifit: {len(tasks)} to run, {skipped_n} skipped, {max_workers} workers")
failed = []

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {ex.submit(_run_dtifit, s): s for s in tasks}
    for fut in tqdm(as_completed(futures), total=len(tasks), desc="dtifit"):
        stem, status = fut.result()
        if "fail" in status:
            failed.append((stem, status))

md_maps = list(dtifit_dir.glob("*/*_MD.nii.gz"))
fa_maps = list(dtifit_dir.glob("*/*_FA.nii.gz"))
print(f"\ndtifit complete: {len(md_maps)} MD maps | {len(fa_maps)} FA maps | {len(failed)} failed")
if failed:
    for s, e in failed[:10]: print(f"  {s}: {e}")

dtifit: 803 to run, 0 skipped, 14 workers


dtifit: 100%|██████████| 803/803 [02:22<00:00,  5.62it/s]


dtifit complete: 787 MD maps | 787 FA maps | 16 failed
  I1092244_114_S_6039: fail:bval_only 0 DWI directions — need ≥6
  I1224877_020_S_6513: fail:bval_only 0 DWI directions — need ≥6
  I1241880_020_S_5203: fail:bval_only 0 DWI directions — need ≥6
  I1278648_020_S_6185: fail:bval_only 0 DWI directions — need ≥6
  I1325865_020_S_6449: fail:bval_only 0 DWI directions — need ≥6
  I1329976_020_S_6513: fail:bval_only 0 DWI directions — need ≥6
  I1332415_020_S_6227: fail:bval_only 0 DWI directions — need ≥6
  I1349126_020_S_6504: fail:bval_only 0 DWI directions — need ≥6
  I1412031_020_S_6901: fail:bval_only 0 DWI directions — need ≥6
  I1428968_022_S_5004: fail:bval_only 0 DWI directions — need ≥6


In [ ]:
# ── SyNRA b0→T1 registration → warp MD→MNI → smooth ─────────────────────────────
import ants, shutil

reg_t1_dir  = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm/dti_long_reg_t1_native")
mni_dir     = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm/dti_long_mni")
smooth_dir  = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm/dti_long_mni_smoothed")
t1_strip_dir = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm/t1_long_skull_strip")
t1_warps_dir = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm/t1_long_vbm_warps")
for d in [reg_t1_dir, mni_dir, smooth_dir]:
    d.mkdir(parents=True, exist_ok=True)

mni_t1   = ants.image_read("model_data/mni_t1.nii")
fwhm_mm  = 5.0
sigma_mm = fwhm_mm / (2.0 * np.sqrt(2.0 * np.log(2.0)))

# dti_stem → t1_stem lookup (adjust to match your actual pairing dataframe)
import pandas as pd
# CORRECT
pair_df   = pd.read_csv("model_data/adni/paired_df_long.csv")
dti_to_t1 = dict(zip(pair_df["dti_image_subject_id"], pair_df["t1_image_subject_id"]))

def _ras_ants(path: Path):
    """Load NIfTI, reorient to RAS, return ANTs image."""
    ras = nib.as_closest_canonical(nib.load(str(path)))
    with tempfile.NamedTemporaryFile(suffix=".nii.gz", delete=False) as f:
        tmp = f.name
    nib.save(ras, tmp)
    out = ants.image_read(tmp)
    os.unlink(tmp)
    return out

def _normalize_for_registration(ants_img):
    """Winsorize to [1, 99] pctile over brain voxels, rescale to [0, 1000].
    No-op on normally-scaled scans; fixes raw-ADU outliers where extreme
    intensity ranges compress the Mattes MI histogram to near-zero gradient."""
    arr = ants_img.numpy()
    brain = arr[arr > 0]
    if brain.size == 0:
        return ants_img
    lo, hi = np.percentile(brain, [1, 99])
    arr = np.clip(arr, lo, hi)
    arr = (arr - lo) / (hi - lo) * 1000.0
    return ants_img.new_image_like(arr)

def _register_warp_smooth(dti_stem: str) -> tuple[str, str]:
    t1_stem  = dti_to_t1.get(dti_stem)
    b0_path  = dti_bet_dir  / f"{dti_stem}_meanb0.nii.gz"
    md_path  = dtifit_dir   / dti_stem / f"{dti_stem}_MD.nii.gz"
    fa_path  = dtifit_dir   / dti_stem / f"{dti_stem}_FA.nii.gz"
    out_sm   = smooth_dir   / f"{dti_stem}_MD_s5.nii.gz"

    if out_sm.exists():
        return dti_stem, "skip"
    if t1_stem is None:
        return dti_stem, "fail:no_t1_pair"
    t1_path  = t1_strip_dir / f"{t1_stem}.nii.gz"
    t1_warp  = t1_warps_dir / f"{t1_stem}_1Warp.nii.gz"
    t1_affine = t1_warps_dir / f"{t1_stem}_0GenericAffine.mat"
    for p in [b0_path, md_path, fa_path, t1_path, t1_warp, t1_affine]:
        if not p.exists():
            return dti_stem, f"fail:missing_{p.name}"

    try:
        fixed  = ants.image_read(str(t1_path))
        moving = _ras_ants(b0_path)

        # Normalize before registration — fixes ADU-scale outliers and neck-remnant
        # BET cases where joint histogram is poorly sampled by Mattes MI.
        # Transforms are purely spatial; apply_transforms below uses originals.
        fixed_norm  = _normalize_for_registration(fixed)
        moving_norm = _normalize_for_registration(moving)

        # ── SyNRA: rigid + affine + SyN deformable (handles EPI distortion) ──
        reg = ants.registration(
            fixed=fixed_norm,
            moving=moving_norm,
            type_of_transform="SyNRA",
            initial_transform="identity",
            aff_metric="mattes",
            syn_metric="mattes",
            syn_sampling=64,
            flow_sigma=3,
            total_sigma=0,
        )

        # save DTI-in-T1 native space maps (warp original images, not normalized)
        for src, out in [(md_path, reg_t1_dir / f"{dti_stem}_MD.nii.gz"),
                         (fa_path, reg_t1_dir / f"{dti_stem}_FA.nii.gz")]:
            warped = ants.apply_transforms(
                fixed=fixed, moving=_ras_ants(src),
                transformlist=reg["fwdtransforms"], interpolator="linear")
            ants.image_write(warped, str(out))

        # save affine component for reference
        tx_mat = reg_t1_dir / f"{dti_stem}_SyNRA.mat"
        for tx in reg["fwdtransforms"]:
            if Path(tx).suffix == ".mat" or "GenericAffine" in tx:
                shutil.copy(tx, tx_mat); break

        # ── warp MD → MNI (chain DTI→T1 SyNRA + T1→MNI SyN) ─────────────────
        md_t1 = ants.image_read(str(reg_t1_dir / f"{dti_stem}_MD.nii.gz"))
        md_mni = ants.apply_transforms(
            fixed=mni_t1, moving=md_t1,
            transformlist=[str(t1_warp), str(t1_affine)],
            interpolator="linear")
        out_mni = mni_dir / f"{dti_stem}_MD.nii.gz"
        ants.image_write(md_mni, str(out_mni))

        # ── 5 mm smooth + scale to µm²/ms ────────────────────────────────────
        smoothed = ants.smooth_image(
            md_mni, sigma=sigma_mm, sigma_in_physical_coordinates=True)
        ants.image_write(smoothed * 1000.0, str(out_sm))

        return dti_stem, "ok"
    except Exception as e:
        return dti_stem, f"fail:{e}"

# only run subjects that have completed dtifit AND a paired T1
ready = [s for s in dti_stems
         if (dtifit_dir / s / f"{s}_MD.nii.gz").exists()
         and s in dti_to_t1
         and not (smooth_dir / f"{s}_MD_s5.nii.gz").exists()]
print(f"SyNRA reg + warp + smooth: {len(ready)} subjects, {max_workers} workers")

failed = []
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {ex.submit(_register_warp_smooth, s): s for s in ready}
    for fut in tqdm(as_completed(futures), total=len(ready), desc="Reg→Warp→Smooth"):
        stem, status = fut.result()
        if "fail" in status:
            failed.append((stem, status))

sm_maps = list(smooth_dir.glob("*_MD_s5.nii.gz"))
print(f"\nDone: {len(sm_maps)} smoothed MD maps in MNI | {len(failed)} failed")
if failed:
    for s, e in failed[:10]: print(f"  {s}: {e}")

In [23]:
vbm_root = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm")
warps_dir = vbm_root / "t1_long_vbm_warps"
jac_dir = vbm_root / "t1_long_jacobian"

mni_t1_path = Path("model_data/mni_t1.nii")
mni_t1 = ants.image_read(str(mni_t1_path))

warp_files = sorted(warps_dir.glob("*_1Warp.nii.gz"))
print(f"Found {len(warp_files)} subject warps to compute Jacobians for")

Found 802 subject warps to compute Jacobians for


In [32]:
vbm_root = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm")
pve_mni_dir = vbm_root / "t1_long_pve_mni"
jac_dir = vbm_root / "t1_long_jacobian"
mod_dir = vbm_root / "t1_long_pve_mni_modulated"

subject_dirs = sorted([p for p in pve_mni_dir.iterdir() if p.is_dir()])
print(f"Found {len(subject_dirs)} subjects to modulate")

Found 802 subjects to modulate


In [33]:
failed = []
skipped = 0

for subj_dir in tqdm(subject_dirs, desc="Modulate PVE * Jacobian"):
    stem = subj_dir.name
    out_subj_dir = mod_dir / stem
    out_subj_dir.mkdir(parents=True, exist_ok=True)

    jac_path = jac_dir / f"{stem}_jacobian.nii.gz"
    if not jac_path.exists():
        failed.append((stem, "missing jacobian"))
        continue

    all_done = all(
        (out_subj_dir / f"{stem}_pve_{i}_mod.nii.gz").exists() for i in (0, 1, 2)
    )
    if all_done:
        skipped += 1
        continue

    try:
        jac_img = ants.image_read(str(jac_path))
        jac_arr = jac_img.numpy()

        for tissue_idx in (0, 1, 2):
            in_pve = subj_dir / f"{stem}_pve_{tissue_idx}_mni.nii.gz"
            out_pve = out_subj_dir / f"{stem}_pve_{tissue_idx}_mod.nii.gz"

            if out_pve.exists():
                continue
            if not in_pve.exists():
                failed.append((stem, f"missing pve_{tissue_idx}_mni"))
                continue

            pve_img = ants.image_read(str(in_pve))
            mod_arr = pve_img.numpy() * jac_arr
            mod_img = ants.from_numpy(
                mod_arr,
                origin=pve_img.origin,
                spacing=pve_img.spacing,
                direction=pve_img.direction,
            )
            ants.image_write(mod_img, str(out_pve))

    except Exception as e:
        failed.append((stem, str(e)))

csf_mod = list(mod_dir.glob("*/*_pve_0_mod.nii.gz"))
gm_mod = list(mod_dir.glob("*/*_pve_1_mod.nii.gz"))
wm_mod = list(mod_dir.glob("*/*_pve_2_mod.nii.gz"))
print(f"\nModulation complete:")
print(f"  {len(csf_mod)} CSF, {len(gm_mod)} GM, {len(wm_mod)} WM modulated PVEs")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Modulate PVE * Jacobian: 100%|██████████| 802/802 [02:22<00:00,  5.61it/s]



Modulation complete:
  802 CSF, 802 GM, 802 WM modulated PVEs
  (0 skipped, already existed)


In [34]:
vbm_root = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm")
mod_dir = vbm_root / "t1_long_pve_mni_modulated"
smooth_dir = vbm_root / "t1_long_pve_mni_smoothed"

fwhm_mm = 2.5
sigma_mm = fwhm_mm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
print(f"Smoothing at {fwhm_mm} mm FWHM (sigma = {sigma_mm:.4f} mm)")

subject_dirs = sorted([p for p in mod_dir.iterdir() if p.is_dir()])
print(f"Found {len(subject_dirs)} subjects to smooth")

Smoothing at 2.5 mm FWHM (sigma = 1.0617 mm)
Found 802 subjects to smooth


In [35]:
failed = []
skipped = 0

for subj_dir in tqdm(subject_dirs, desc="Smooth modulated PVEs"):
    stem = subj_dir.name
    out_subj_dir = smooth_dir / stem
    out_subj_dir.mkdir(parents=True, exist_ok=True)

    all_done = all(
        (out_subj_dir / f"{stem}_pve_{i}_mod_s2p5.nii.gz").exists() for i in (0, 1, 2)
    )
    if all_done:
        skipped += 1
        continue

    try:
        for tissue_idx in (0, 1, 2):
            in_pve = subj_dir / f"{stem}_pve_{tissue_idx}_mod.nii.gz"
            out_pve = out_subj_dir / f"{stem}_pve_{tissue_idx}_mod_s2p5.nii.gz"

            if out_pve.exists():
                continue
            if not in_pve.exists():
                failed.append((stem, f"missing pve_{tissue_idx}_mod"))
                continue

            img = ants.image_read(str(in_pve))
            smoothed = ants.smooth_image(
                img,
                sigma=sigma_mm,
                sigma_in_physical_coordinates=True,
            )
            ants.image_write(smoothed, str(out_pve))

    except Exception as e:
        failed.append((stem, str(e)))

csf_s = list(smooth_dir.glob("*/*_pve_0_mod_s2p5.nii.gz"))
gm_s = list(smooth_dir.glob("*/*_pve_1_mod_s2p5.nii.gz"))
wm_s = list(smooth_dir.glob("*/*_pve_2_mod_s2p5.nii.gz"))
print(f"\nSmoothing complete:")
print(f"  {len(csf_s)} CSF, {len(gm_s)} GM, {len(wm_s)} WM smoothed maps")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Smooth modulated PVEs: 100%|██████████| 802/802 [02:45<00:00,  4.84it/s]



Smoothing complete:
  802 CSF, 802 GM, 802 WM smoothed maps
  (0 skipped, already existed)


In [27]:
failed = []
skipped = 0

for warp_path in tqdm(warp_files, desc="Jacobian (warp * affine)"):
    stem = warp_path.name.replace("_1Warp.nii.gz", "")
    affine_path = warps_dir / f"{stem}_0GenericAffine.mat"
    out_jac = jac_dir / f"{stem}_jacobian.nii.gz"

    if out_jac.exists():
        skipped += 1
        continue

    if not affine_path.exists():
        failed.append((stem, "missing affine"))
        continue

    try:
        jac_warp = ants.create_jacobian_determinant_image(
            domain_image=mni_t1,
            tx=str(warp_path),
            do_log=False,
            geom=False,
        )

        affine_tx = ants.read_transform(str(affine_path))
        params = np.asarray(affine_tx.parameters)
        A = params[:9].reshape(3, 3)
        det_affine = float(abs(np.linalg.det(A)))

        jac_total = jac_warp * det_affine
        ants.image_write(jac_total, str(out_jac))

    except Exception as e:
        failed.append((stem, str(e)))

jac_maps = list(jac_dir.glob("*_jacobian.nii.gz"))
print(f"\nJacobian computation complete: {len(jac_maps)} maps")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Jacobian (warp * affine): 100%|██████████| 802/802 [03:12<00:00,  4.16it/s]


Jacobian computation complete: 802 maps
  (0 skipped, already existed)


In [30]:
vbm_root = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm")
seg_dir = vbm_root / "t1_long_seg_native"
warps_dir = vbm_root / "t1_long_vbm_warps"
pve_mni_dir = vbm_root / "t1_long_pve_mni"

mni_t1_path = Path("model_data/mni_t1.nii")
mni_t1 = ants.image_read(str(mni_t1_path))

subject_dirs = sorted([p for p in seg_dir.iterdir() if p.is_dir()])
print(f"Found {len(subject_dirs)} segmented subjects to warp into MNI")

Found 802 segmented subjects to warp into MNI


In [31]:
failed = []
skipped = 0

for subj_dir in tqdm(subject_dirs, desc="Warp PVEs -> MNI"):
    stem = subj_dir.name
    out_subj_dir = pve_mni_dir / stem
    out_subj_dir.mkdir(parents=True, exist_ok=True)

    affine = warps_dir / f"{stem}_0GenericAffine.mat"
    warp = warps_dir / f"{stem}_1Warp.nii.gz"

    if not affine.exists() or not warp.exists():
        failed.append((stem, "missing transforms"))
        continue

    transformlist = [str(warp), str(affine)]

    all_done = True
    for tissue_idx in (0, 1, 2):
        out_pve = out_subj_dir / f"{stem}_pve_{tissue_idx}_mni.nii.gz"
        if not out_pve.exists():
            all_done = False
            break
    if all_done:
        skipped += 1
        continue

    try:
        for tissue_idx in (0, 1, 2):
            in_pve = subj_dir / f"{stem}_pve_{tissue_idx}.nii.gz"
            out_pve = out_subj_dir / f"{stem}_pve_{tissue_idx}_mni.nii.gz"

            if out_pve.exists():
                continue
            if not in_pve.exists():
                failed.append((stem, f"missing pve_{tissue_idx}"))
                continue

            moving = ants.image_read(str(in_pve))
            warped = ants.apply_transforms(
                fixed=mni_t1,
                moving=moving,
                transformlist=transformlist,
                interpolator="linear",
            )
            ants.image_write(warped, str(out_pve))

    except Exception as e:
        failed.append((stem, str(e)))

csf_maps = list(pve_mni_dir.glob("*/*_pve_0_mni.nii.gz"))
gm_maps = list(pve_mni_dir.glob("*/*_pve_1_mni.nii.gz"))
wm_maps = list(pve_mni_dir.glob("*/*_pve_2_mni.nii.gz"))
print(f"\nWarp PVEs to MNI complete:")
print(f"  {len(csf_maps)} CSF, {len(gm_maps)} GM, {len(wm_maps)} WM in MNI space")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Warp PVEs -> MNI: 100%|██████████| 802/802 [09:04<00:00,  1.47it/s]


Warp PVEs to MNI complete:
  802 CSF, 802 GM, 802 WM in MNI space
  (0 skipped, already existed)


In [10]:
vbm_root = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm")
raw_dir = vbm_root / "t1_long_raw"
warps_dir = vbm_root / "t1_long_vbm_warps"
warps_dir.mkdir(parents=True, exist_ok=True)

mni_t1_path = Path("model_data/mni_t1.nii")
mni_t1 = ants.image_read(str(mni_t1_path))

raw_files = sorted(raw_dir.glob("*.nii"))
print(f"Found {len(raw_files)} raw native T1 images to register")

Found 802 raw native T1 images to register


In [ ]:
# ── Force-rerun bad T1→MNI registrations ────────────────────────────────────
# Populate with stems whose existing warp outputs are known-bad.
# Running this cell deletes those outputs so cell 18 re-registers them.
force_rerun_stems = [
    "I1039134_014_S_6437",
    "I1481579_014_S_6988",
    "I1170887_141_S_6015",
    "I1431746_070_S_6236",
    "I491256_002_S_1280",
    # add remaining bad stems here
]

suffixes = [
    "_warpedT1.nii.gz",
    "_0GenericAffine.mat",
    "_1Warp.nii.gz",
    "_1InverseWarp.nii.gz",
]

deleted = 0
for stem in force_rerun_stems:
    for suffix in suffixes:
        p = warps_dir / f"{stem}{suffix}"
        if p.exists():
            p.unlink()
            deleted += 1
print(f"Cleared {deleted} files for {len(force_rerun_stems)} subjects — re-run cell 18 to reregister.")

In [11]:
def _normalize_for_registration(ants_img):
    """Winsorize to [1, 99] pctile over brain voxels, rescale to [0, 1000].
    No-op on normally-scaled scans; fixes raw-ADU outliers where extreme
    intensity ranges compress the Mattes MI histogram to near-zero gradient."""
    arr = ants_img.numpy()
    brain = arr[arr > 0]
    if brain.size == 0:
        return ants_img
    lo, hi = np.percentile(brain, [1, 99])
    arr = np.clip(arr, lo, hi)
    arr = (arr - lo) / (hi - lo) * 1000.0
    return ants_img.new_image_like(arr)

# Pre-normalize MNI template once; re-used for every subject
mni_norm = _normalize_for_registration(mni_t1)

failed = []
skipped = 0

for raw_path in tqdm(raw_files, desc="SyN raw T1 -> MNI"):
    stem = raw_path.stem
    out_warped = warps_dir / f"{stem}_warpedT1.nii.gz"
    out_affine = warps_dir / f"{stem}_0GenericAffine.mat"
    out_warp = warps_dir / f"{stem}_1Warp.nii.gz"
    out_inv_warp = warps_dir / f"{stem}_1InverseWarp.nii.gz"

    if (
        out_warped.exists()
        and out_affine.exists()
        and out_warp.exists()
        and out_inv_warp.exists()
    ):
        skipped += 1
        continue

    try:
        moving = ants.image_read(str(raw_path))

        # Normalize before registration — fixes ADU-scale outliers and neck-remnant
        # BET cases. Transforms are spatial; re-apply to original for native intensities.
        moving_norm = _normalize_for_registration(moving)
        reg = ants.registration(fixed=mni_norm, moving=moving_norm, type_of_transform="SyN")

        # Warp original (unnormalized) moving to preserve native T1 intensities
        warped_native = ants.apply_transforms(
            fixed=mni_t1, moving=moving,
            transformlist=reg["fwdtransforms"], interpolator="linear")
        ants.image_write(warped_native, str(out_warped))

        for tx_path in reg["fwdtransforms"]:
            tx = Path(tx_path)
            if "GenericAffine" in tx.name:
                shutil.copy(tx, out_affine)
            elif "Warp" in tx.name and "Inverse" not in tx.name:
                shutil.copy(tx, out_warp)

        for tx_path in reg["invtransforms"]:
            tx = Path(tx_path)
            if "InverseWarp" in tx.name:
                shutil.copy(tx, out_inv_warp)

    except Exception as e:
        failed.append((stem, str(e)))

warped = list(warps_dir.glob("*_warpedT1.nii.gz"))
affines = list(warps_dir.glob("*_0GenericAffine.mat"))
fwd_warps = list(warps_dir.glob("*_1Warp.nii.gz"))
inv_warps = list(warps_dir.glob("*_1InverseWarp.nii.gz"))
print(f"\nVBM SyN registration complete:")
print(f"  {len(warped)} warped T1s")
print(f"  {len(affines)} affines, {len(fwd_warps)} forward warps, {len(inv_warps)} inverse warps")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

SyN raw T1 -> MNI: 100%|██████████| 802/802 [56:10<00:00,  4.20s/it]  


VBM SyN registration complete:
  802 warped T1s
  802 affines, 802 forward warps, 802 inverse warps
  (0 skipped, already existed)


#### DTI: DICOM → NIfTI → BET → dtifit (long cohort)

Format-aware conversion handles all 5 ADNI DTI DICOM types:
- **Siemens MOSAIC** (425 subjects): bval in CSA `(0019,100C)` — dcm2niix reads automatically
- **GE classic** (58): bval in `(0043,1039)`, b=0 may appear as -1 in standard tag — dcm2niix corrects
- **Philips enhanced** (12): 1 DCM file / 2880 frames, SOPClassUID enhanced MR
- **Philips classic** (36) + **GE MPTronic** (1): standard conversion

Key flags: `-m y` merges 2D slices (required for GE/Philips classic); `-i y` drops derived/localizer series that contaminate many-file datasets with high-intensity navigator echoes.

In [3]:
# --- DTI DCM → NIfTI (long cohort) -------------------------------------------
import re

dti_dcm_root = Path("model_data/adni/dti_long_data/dti_long_dcm")
dti_vbm_root = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm")
dti_raw_dir  = Path("model_data/adni/dti_long_data/dti_long_raw")

_IMAGE_DIR = re.compile(r"^I\d+$")


def _dti_scan_dirs(dcm_root: Path) -> list[Path]:
    """All I<digits> folders under dcm_root that contain .dcm files."""
    dirs = []
    for p in sorted(dcm_root.rglob("*")):
        if p.is_dir() and _IMAGE_DIR.fullmatch(p.name):
            if any(p.glob("*.dcm")) or any(p.glob("*.DCM")):
                dirs.append(p)
    return dirs


def classify_dti_dcm(dcm_dir: Path) -> str:
    """Returns: 'siemens_mosaic' | 'ge_classic' | 'philips_enhanced' | 'philips_classic'"""
    files = [f for f in dcm_dir.iterdir() if not f.name.startswith(".")]
    if not files:
        return "unknown"
    if len(files) == 1:
        ds = pydicom.dcmread(str(files[0]), stop_before_pixels=True)
        if str(getattr(ds, "SOPClassUID", "")) == "1.2.840.10008.5.1.4.1.1.4.1":
            return "philips_enhanced"
    ds = pydicom.dcmread(str(files[0]), stop_before_pixels=True)
    img_type = list(getattr(ds, "ImageType", []))
    mfr = getattr(ds, "Manufacturer", "").upper()
    if "MOSAIC" in img_type:
        return "siemens_mosaic"
    if "GE" in mfr or "MPTRONIC" in mfr:
        return "ge_classic"
    return "philips_classic"


scan_dirs = _dti_scan_dirs(dti_dcm_root)
print(f"Found {len(scan_dirs)} DTI image folders")

Found 802 DTI image folders


In [4]:
failed    = []
skipped   = 0
fmt_counts: dict[str, int] = {}

for scan_dir in tqdm(scan_dirs, desc="dcm2niix DTI"):
    subj = scan_dir.parent.parent.parent.name   # ADNI PTID
    stem = f"{scan_dir.name}_{subj}"

    if (dti_raw_dir / f"{stem}.nii.gz").exists():
        skipped += 1
        continue

    fmt = classify_dti_dcm(scan_dir)
    fmt_counts[fmt] = fmt_counts.get(fmt, 0) + 1

    r = subprocess.run(
        [
            "dcm2niix",
            "-m", "y",   # merge 2D slices → 4D (essential for GE/Philips classic)
            "-b", "y",   # write bval/bvec + JSON sidecar
            "-z", "y",   # gzip output
            "-i", "y",   # ignore derived/localizer series (drops navigator echoes)
            "-f", stem,
            "-o", str(dti_raw_dir),
            str(scan_dir),
        ],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        failed.append((stem, fmt, r.stderr.strip()[:120]))
        continue

    # dcm2niix writes a JSON sidecar alongside bval/bvec when -b y is set.
    # The JSON contains acquisition metadata (EffectiveEchoSpacing, TotalReadoutTime,
    # ImageOrientationPatientDICOM, bval source tag) needed for eddy/topup and provenance.
    # If dcm2niix emitted a differently-stemmed JSON (series split), copy the first match.
    json_dst = dti_raw_dir / f"{stem}.json"
    if not json_dst.exists():
        candidates = sorted(dti_raw_dir.glob(f"{scan_dir.name}_*.json"))
        if candidates:
            shutil.copy(candidates[0], json_dst)

nii   = list(dti_raw_dir.glob("I*.nii.gz"))
jsons = list(dti_raw_dir.glob("I*.json"))
print(f"DTI NIfTI: {len(nii)} | JSON sidecars: {len(jsons)} "
      f"({skipped} skipped, {len(failed)} failed)")
print(f"Format breakdown (new conversions): {fmt_counts}")
if failed:
    for name, fmt, err in failed[:5]:
        print(f"  [{fmt}] {name}: {err}")

dcm2niix DTI: 100%|██████████| 802/802 [1:38:58<00:00,  7.40s/it]  

DTI NIfTI: 804 | JSON sidecars: 804 (0 skipped, 0 failed)
Format breakdown (new conversions): {'siemens_mosaic': 695, 'ge_classic': 60, 'philips_classic': 35, 'philips_enhanced': 12}


In [ ]:
# --- bval/bvec QC + exclusion set + stale dtifit cleanup ---------------------
# check_bval_adequacy uses b < 50 for b=0 and b >= 500 for DWI — more specific
# than a b < 100 / b >= 100 split, correct for ADNI's b=1000 protocol.

_primary_bval = re.compile(r"^I\d+_\d{3}_S_\d+\.bval$")

def check_bval_adequacy(bval_path: Path) -> tuple[bool, str]:
    bvals = np.loadtxt(str(bval_path)).flatten()
    n_b0  = int(np.sum(bvals < 50))
    n_dwi = int(np.sum(bvals >= 500))
    if n_b0 == 0:
        return False, "no b=0 volumes"
    if n_dwi < 6:
        return False, f"only {n_dwi} DWI directions — need ≥6 for valid tensor"
    return True, f"ok ({n_b0} b=0, {n_dwi} DWI)"


dtifit_dir    = dti_vbm_root / "dti_long_ditfit"
dti_strip_dir = dti_vbm_root / "dti_long_skull_strip"
md_t1_dir     = dti_vbm_root / "dti_long_reg_t1_native"
md_mni_dir    = dti_vbm_root / "dti_long_mni"
md_smooth_dir = dti_vbm_root / "dti_long_mni_smoothed"
dti_strip_dir.mkdir(parents=True, exist_ok=True)

bval_files = sorted(f for f in dti_raw_dir.glob("*.bval") if _primary_bval.match(f.name))
print(f"Checking {len(bval_files)} bval files ...")

dti_long_exclude: set[str] = set()
issues: list[tuple[str, str]] = []

for bval_path in bval_files:
    stem = bval_path.stem
    bvec_path = bval_path.with_suffix(".bvec")
    nii_path  = dti_raw_dir / f"{stem}.nii.gz"

    # bvec must exist
    if not bvec_path.exists():
        issues.append((stem, "missing_bvec"))
        dti_long_exclude.add(stem)
        continue

    # Check adequacy
    ok, msg = check_bval_adequacy(bval_path)
    if not ok:
        issues.append((stem, msg))
        dti_long_exclude.add(stem)
        continue

    # Volume count must match bval count
    if nii_path.exists():
        try:
            shape = nib.load(str(nii_path)).shape
            n_vols = shape[3] if len(shape) == 4 else 1
            bvals  = np.loadtxt(str(bval_path)).flatten()
            if n_vols != len(bvals):
                issues.append((stem, f"vol_bval_mismatch nifti={n_vols} bval={len(bvals)}"))
                dti_long_exclude.add(stem)
        except Exception as e:
            issues.append((stem, f"nifti_read_error: {e}"))

print(f"Excluded (unfittable): {len(dti_long_exclude)}")
for s in sorted(dti_long_exclude):
    reason = next((r for st, r in issues if st == s), "?")
    print(f"  EXCLUDE {s}: {reason}")

# Remove stale dtifit / downstream outputs for excluded subjects so they are
# cleanly absent from the MD map glob in cell e065e153.
n_cleaned = 0
for stem in dti_long_exclude:
    for path in [
        dtifit_dir / stem / f"{stem}_FA.nii.gz",
        dtifit_dir / stem / f"{stem}_MD.nii.gz",
        md_t1_dir  / f"{stem}_FA.nii.gz",
        md_t1_dir  / f"{stem}_MD.nii.gz",
        md_t1_dir  / f"{stem}_rigid.mat",
        md_mni_dir / f"{stem}_MD.nii.gz",
        md_smooth_dir / f"{stem}_MD_s5.nii.gz",
    ]:
        if path.exists():
            path.unlink()
            n_cleaned += 1
print(f"Cleaned {n_cleaned} stale outputs for excluded subjects")

In [ ]:
# --- BET (DTI long cohort): extract first b=0, skull-strip, apply mask -------
_primary_nii = re.compile(r"^I\d+_\d{3}_S_\d+\.nii\.gz$")
dti_raw_files = sorted(f for f in dti_raw_dir.glob("*.nii.gz") if _primary_nii.match(f.name))

if "dti_long_exclude" not in dir():
    dti_long_exclude = set()
    print("WARNING: dti_long_exclude not in scope — run QC cell above first")

dti_raw_files = [f for f in dti_raw_files if f.stem.replace(".nii", "") not in dti_long_exclude]
print(f"BET on {len(dti_raw_files)} DTI volumes "
      f"(bet -m -f 0.3, {len(dti_long_exclude)} excluded by QC)")

failed  = []
skipped = 0

for dti_path in tqdm(dti_raw_files, desc="BET DTI long"):
    stem = dti_path.name.replace(".nii.gz", "")
    out_nii   = dti_strip_dir / f"{stem}.nii.gz"
    mask_path = dti_strip_dir / f"{stem}_mask.nii.gz"
    bval_dst  = dti_strip_dir / f"{stem}.bval"
    bvec_dst  = dti_strip_dir / f"{stem}.bvec"

    if out_nii.exists() and mask_path.exists() and bval_dst.exists() and bvec_dst.exists():
        skipped += 1
        continue

    bval_src = dti_raw_dir / f"{stem}.bval"
    bvec_src = dti_raw_dir / f"{stem}.bvec"

    # Find the actual first b=0 volume index from bvals
    b0_idx = 0
    if bval_src.exists():
        try:
            _bv = np.loadtxt(str(bval_src)).flatten()
            _idxs = np.where(_bv < 50)[0]
            b0_idx = int(_idxs[0]) if len(_idxs) > 0 else 0
        except Exception:
            b0_idx = 0

    b0_path       = dti_strip_dir / f"{stem}_b0.nii.gz"
    b0_brain_base = dti_strip_dir / f"{stem}_b0_brain"
    b0_brain      = dti_strip_dir / f"{stem}_b0_brain.nii.gz"
    b0_mask       = dti_strip_dir / f"{stem}_b0_brain_mask.nii.gz"

    try:
        subprocess.run(
            ["fslroi", str(dti_path), str(b0_path), str(b0_idx), "1"],
            capture_output=True, text=True, check=True,
        )
        subprocess.run(
            ["bet", str(b0_path), str(b0_brain_base), "-m", "-f", "0.3"],
            capture_output=True, text=True, check=True,
        )
        subprocess.run(
            ["fslmaths", str(dti_path), "-mas", str(b0_mask), str(out_nii)],
            capture_output=True, text=True, check=True,
        )
        shutil.move(str(b0_mask), str(mask_path))
        if bval_src.exists(): shutil.copy(bval_src, bval_dst)
        if bvec_src.exists(): shutil.copy(bvec_src, bvec_dst)
        for tmp in [b0_path, b0_brain]:
            if tmp.exists(): tmp.unlink()
    except subprocess.CalledProcessError as e:
        failed.append((stem, e.stderr.strip() if e.stderr else str(e)))
    except Exception as e:
        failed.append((stem, str(e)))

print(f"BET complete: {len(list(dti_strip_dir.glob('I*_*_S_*.nii.gz')))} stripped | "
      f"{skipped} skipped | {len(dti_long_exclude)} excluded | {len(failed)} failed")
if failed:
    for name, err in failed[:5]:
        print(f"  {name}: {err}")

In [ ]:
# --- dtifit (long cohort) -----------------------------------------------------
# Skip-logic: existing FA in dti_long_ditfit is preserved (801 subjects already done).
# Only subjects with a newly skull-stripped NIfTI that don't have FA yet are fitted.
# Excluded subjects have no stripped NIfTI (BET above skipped them) so they won't appear.
_dtifit_pat = re.compile(r"^I\d+_\d{3}_S_\d+\.nii\.gz$")
strip_files = sorted(f for f in dti_strip_dir.glob("*.nii.gz") if _dtifit_pat.match(f.name))

if "dti_long_exclude" not in dir():
    dti_long_exclude = set()
    print("WARNING: dti_long_exclude not in scope — run QC cell above first")

strip_files = [f for f in strip_files if f.name.replace(".nii.gz", "") not in dti_long_exclude]
print(f"dtifit on {len(strip_files)} skull-stripped DTI volumes "
      f"({len(dti_long_exclude)} excluded by QC)")

failed  = []
skipped = 0

for dti_path in tqdm(strip_files, desc="dtifit long"):
    stem = dti_path.name.replace(".nii.gz", "")
    mask_path = dti_strip_dir / f"{stem}_mask.nii.gz"
    bval_path = dti_strip_dir / f"{stem}.bval"
    bvec_path = dti_strip_dir / f"{stem}.bvec"

    subj_dir = dtifit_dir / stem
    subj_dir.mkdir(parents=True, exist_ok=True)
    fa_file  = subj_dir / f"{stem}_FA.nii.gz"

    if fa_file.exists():
        skipped += 1
        continue

    missing = [p for p in [mask_path, bval_path, bvec_path] if not p.exists()]
    if missing:
        failed.append((stem, f"missing: {[p.name for p in missing]}"))
        continue

    r = subprocess.run(
        [
            "dtifit",
            "-k", str(dti_path),
            "-m", str(mask_path),
            "-r", str(bvec_path),
            "-b", str(bval_path),
            "-o", str(subj_dir / stem),
        ],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        failed.append((stem, r.stderr.strip()[:120]))

fa_maps = list(dtifit_dir.glob("*/*_FA.nii.gz"))
md_maps = list(dtifit_dir.glob("*/*_MD.nii.gz"))
print(f"dtifit complete: {len(fa_maps)} FA | {len(md_maps)} MD | "
      f"{skipped} skipped | {len(dti_long_exclude)} excluded | {len(failed)} failed")
if failed:
    for name, err in failed[:5]:
        print(f"  {name}: {err}")

In [36]:
import re
from concurrent.futures import ThreadPoolExecutor, as_completed

t1_strip_dir = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm/t1_long_skull_strip")
t1_warps_dir = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm/t1_long_vbm_warps")

dti_vbm_root = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm")
dtifit_dir = dti_vbm_root / "dti_long_ditfit"
md_t1_dir = dti_vbm_root / "dti_long_reg_t1_native"

subj_pat = re.compile(r"^I\d+_(\d{3}_S_\d+)$")

md_maps = sorted(dtifit_dir.glob("*/*_MD.nii.gz"))
print(f"Found {len(md_maps)} DTI MD maps")

Found 801 DTI MD maps


In [75]:
import re
import pandas as pd
from pathlib import Path

t1_dcm_root  = Path("model_data/adni/t1_long_data/t1_long_dcm")
dti_dcm_root = Path("model_data/adni/dti_long_data/dti_long_dcm")
_date_pat = re.compile(r"^(\d{4}-\d{2}-\d{2})")
_img_pat  = re.compile(r"^I?(\d+)$")

def _build_date_map(dcm_root):
    mapping = {}
    for subj_dir in dcm_root.iterdir():
        if not subj_dir.is_dir(): continue
        for mod_dir in subj_dir.iterdir():
            if not mod_dir.is_dir(): continue
            for date_dir in mod_dir.iterdir():
                if not date_dir.is_dir(): continue
                m = _date_pat.match(date_dir.name)
                if not m: continue
                scan_date = pd.Timestamp(m.group(1))
                for img_dir in date_dir.iterdir():
                    if not img_dir.is_dir(): continue
                    im = _img_pat.match(img_dir.name)
                    if im:
                        mapping[im.group(1)] = scan_date
    return mapping

# stem -> date: extract numeric image_id from "I<id>_<subject>" stem
def _stem_to_date(stem, date_map):
    img_id = stem.lstrip("I").split("_")[0]
    return date_map.get(img_id)

t1_date_raw  = _build_date_map(t1_dcm_root)
dti_date_raw = _build_date_map(dti_dcm_root)

# Wrap as stem-keyed dicts for use in the matching loop below
t1_date  = {f.name.replace(".nii.gz", ""): _stem_to_date(f.name.replace(".nii.gz", ""), t1_date_raw)
            for f in sorted(t1_strip_dir.glob("*.nii.gz"))}
dti_date = {subj_dir.name: _stem_to_date(subj_dir.name, dti_date_raw)
            for subj_dir in sorted(dtifit_dir.iterdir()) if subj_dir.is_dir()}

print(f"t1_date:  {sum(v is not None for v in t1_date.values())}/{len(t1_date)} resolved")
print(f"dti_date: {sum(v is not None for v in dti_date.values())}/{len(dti_date)} resolved")

# --- Match each DTI scan to the closest-date T1 for the same subject ----------
t1_by_subj = {}
for f in sorted(t1_strip_dir.glob("*.nii.gz")):
    stem = f.name.replace(".nii.gz", "")
    m = subj_pat.match(stem)
    if m:
        t1_by_subj.setdefault(m.group(1), []).append(stem)

dti_by_subj = {}
for md_path in md_maps:
    dti_stem = md_path.name.replace("_MD.nii.gz", "")
    m = subj_pat.match(dti_stem)
    if m:
        dti_by_subj.setdefault(m.group(1), []).append((dti_stem, md_path))

dti_to_t1 = {}
unmatched = []
for subj, dti_list in dti_by_subj.items():
    t1_list = sorted(t1_by_subj.get(subj, []))
    if not t1_list:
        continue
    for dti_stem, _ in sorted(dti_list, key=lambda x: x[0]):
        dti_dt = dti_date.get(dti_stem)
        if dti_dt is None:
            # fallback: sort-order pairing
            idx = sorted(dti_list, key=lambda x: x[0]).index((dti_stem, _))
            dti_to_t1[dti_stem] = t1_list[min(idx, len(t1_list) - 1)]
            continue
        t1_with_dates = [(t, t1_date.get(t)) for t in t1_list]
        t1_with_dates = [(t, d) for t, d in t1_with_dates if d is not None]
        if not t1_with_dates:
            dti_to_t1[dti_stem] = t1_list[0]
        else:
            dti_to_t1[dti_stem] = min(t1_with_dates, key=lambda x: abs(x[1] - dti_dt))[0]

print(f"Matched {len(dti_to_t1)} DTI scans to T1 scans")

t1_date:  802/802 resolved
dti_date: 801/801 resolved
Matched 801 DTI scans to T1 scans


In [88]:
def _rigid_md_to_t1(md_path):
    dti_stem = md_path.name.replace("_MD.nii.gz", "")
    t1_stem = dti_to_t1.get(dti_stem)
    fa_path = md_path.parent / f"{dti_stem}_FA.nii.gz"

    if t1_stem is None:
        return dti_stem, "fail", "no matching T1 for subject"

    out_fa  = md_t1_dir / f"{dti_stem}_FA.nii.gz"
    out_md  = md_t1_dir / f"{dti_stem}_MD.nii.gz"
    out_tx  = md_t1_dir / f"{dti_stem}_rigid.mat"

    if out_md.exists() and out_tx.exists():
        return dti_stem, "skip", ""

    t1_path = t1_strip_dir / f"{t1_stem}.nii.gz"
    if not t1_path.exists():
        return dti_stem, "fail", f"missing T1: {t1_path.name}"
    for p in [fa_path, md_path]:
        if not p.exists():
            return dti_stem, "fail", f"missing {p.name}"

    try:
        fixed = ants.image_read(str(t1_path))

        # Reorient FA and MD to canonical RAS before registration.
        # Some acquisitions are stored in LAS, causing ~200 mm origin mismatches
        # that defeat identity-initialized rigid registration.
        import nibabel as _nib, tempfile as _tmp, os as _os

        def _to_ras_ants(path):
            img = _nib.load(str(path))
            img_ras = _nib.as_closest_canonical(img)
            with _tmp.NamedTemporaryFile(suffix=".nii.gz", delete=False) as f:
                tmp = f.name
            _nib.save(img_ras, tmp)
            out = ants.image_read(tmp)
            _os.unlink(tmp)
            return out

        moving_fa = _to_ras_ants(fa_path)
        moving_md = _to_ras_ants(md_path)

        reg = ants.registration(
            fixed=fixed,
            moving=moving_fa,
            type_of_transform="Rigid",
            initial_transform="identity",
            aff_metric="mattes",
        )

        ants.image_write(reg["warpedmovout"], str(out_fa))

        tx_saved = False
        for tx in reg["fwdtransforms"]:
            tx_p = Path(tx)
            if tx_p.suffix == ".mat" or "GenericAffine" in tx_p.name:
                shutil.copy(tx, out_tx)
                tx_saved = True
                break
        if not tx_saved:
            return dti_stem, "fail", "no .mat in fwdtransforms"

        md_in_t1 = ants.apply_transforms(
            fixed=fixed, moving=moving_md,
            transformlist=reg["fwdtransforms"],
            interpolator="linear",
        )
        ants.image_write(md_in_t1, str(out_md))
        return dti_stem, "ok", ""

    except Exception as e:
        return dti_stem, "fail", str(e)

In [89]:
failed = []
skipped = 0
ok = 0

max_workers = max(1, (os.cpu_count() or 4) - 2)
print(f"Running rigid MD -> native T1 with {max_workers} workers")

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(_rigid_md_to_t1, p) for p in md_maps]
    for fut in tqdm(as_completed(futures), total=len(md_maps), desc="Rigid MD -> T1"):
        stem, status, err = fut.result()
        if status == "ok":
            ok += 1
        elif status == "skip":
            skipped += 1
        else:
            failed.append((stem, err))

reg_files = list(md_t1_dir.glob("*_MD.nii.gz"))
print(f"\nRigid MD -> native T1 complete:")
print(f"  {len(reg_files)} MD maps in native T1 space")
print(f"  ({ok} new, {skipped} skipped, {len(failed)} failed)")
if failed:
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Running rigid MD -> native T1 with 14 workers


Rigid MD -> T1:  15%|█▍        | 119/801 [05:23<32:45,  2.88s/it] /Users/runner/work/ANTsPy/ANTsPy/itksource/Modules/ThirdParty/VNL/src/vxl/core/vnl/algo/vnl_svd.hxx: suspicious return value (2) from SVDC
/Users/runner/work/ANTsPy/ANTsPy/itksource/Modules/ThirdParty/VNL/src/vxl/core/vnl/algo/vnl_svd.hxx: M is 2x2
M = [ ...
 0.0214416086674              nan 
               0              nan  ]
/Users/runner/work/ANTsPy/ANTsPy/itksource/Modules/ThirdParty/VNL/src/vxl/core/vnl/algo/vnl_svd.hxx: suspicious return value (2) from SVDC
/Users/runner/work/ANTsPy/ANTsPy/itksource/Modules/ThirdParty/VNL/src/vxl/core/vnl/algo/vnl_svd.hxx: M is 2x2
M = [ ...
 0.0214416086674              nan 
               0              nan  ]
/Users/runner/work/ANTsPy/ANTsPy/itksource/Modules/ThirdParty/VNL/src/vxl/core/vnl/algo/vnl_svd.hxx: suspicious return value (2) from SVDC
/Users/runner/work/ANTsPy/ANTsPy/itksource/Modules/ThirdParty/VNL/src/vxl/core/vnl/algo/vnl_svd.hxx: M is 2x2
M = [ ...
 0.0214416086


Rigid MD -> native T1 complete:
  801 MD maps in native T1 space
  (801 new, 0 skipped, 0 failed)


In [78]:
from concurrent.futures import ThreadPoolExecutor, as_completed

t1_warps_dir = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm/t1_long_vbm_warps")
mni_t1_path = Path("model_data/mni_t1.nii")
mni_t1 = ants.image_read(str(mni_t1_path))

dti_vbm_root = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm")
md_t1_dir = dti_vbm_root / "dti_long_reg_t1_native"
md_mni_dir = dti_vbm_root / "dti_long_mni"

md_t1_files = sorted(md_t1_dir.glob("*_MD.nii.gz"))
print(f"Found {len(md_t1_files)} MD maps in native T1 space to warp into MNI")

Found 801 MD maps in native T1 space to warp into MNI


In [79]:
def _warp_md_to_mni(md_t1_path):
    dti_stem = md_t1_path.name.replace("_MD.nii.gz", "")
    t1_stem = dti_to_t1.get(dti_stem)
    if t1_stem is None:
        return dti_stem, "fail", "no T1 mapping"

    out_md = md_mni_dir / f"{dti_stem}_MD.nii.gz"
    if out_md.exists():
        return dti_stem, "skip", ""

    affine = t1_warps_dir / f"{t1_stem}_0GenericAffine.mat"
    warp = t1_warps_dir / f"{t1_stem}_1Warp.nii.gz"
    missing = [p for p in [affine, warp] if not p.exists()]
    if missing:
        return dti_stem, "fail", f"missing warps: {[p.name for p in missing]}"

    try:
        moving = ants.image_read(str(md_t1_path))
        warped = ants.apply_transforms(
            fixed=mni_t1,
            moving=moving,
            transformlist=[str(warp), str(affine)],
            interpolator="linear",
        )
        ants.image_write(warped, str(out_md))
        return dti_stem, "ok", ""
    except Exception as e:
        return dti_stem, "fail", str(e)

In [80]:
failed = []
skipped = 0
ok = 0

max_workers = max(1, (os.cpu_count() or 4) - 2)
print(f"Running apply_transforms MD -> MNI with {max_workers} workers")

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(_warp_md_to_mni, p) for p in md_t1_files]
    for fut in tqdm(as_completed(futures), total=len(md_t1_files), desc="Warp MD -> MNI"):
        stem, status, err = fut.result()
        if status == "ok":
            ok += 1
        elif status == "skip":
            skipped += 1
        else:
            failed.append((stem, err))

mni_files = list(md_mni_dir.glob("*_MD.nii.gz"))
print(f"\nWarp MD -> MNI complete:")
print(f"  {len(mni_files)} MD maps in MNI space")
print(f"  ({ok} new, {skipped} skipped, {len(failed)} failed)")
if failed:
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Running apply_transforms MD -> MNI with 14 workers


Warp MD -> MNI: 100%|██████████| 801/801 [03:09<00:00,  4.23it/s]


Warp MD -> MNI complete:
  801 MD maps in MNI space
  (801 new, 0 skipped, 0 failed)


In [81]:
from concurrent.futures import ThreadPoolExecutor, as_completed

dti_vbm_root = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm")
md_mni_dir = dti_vbm_root / "dti_long_mni"
md_smooth_dir = dti_vbm_root / "dti_long_mni_smoothed"

fwhm_mm = 5.0
sigma_mm = fwhm_mm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
print(f"Smoothing MD at {fwhm_mm} mm FWHM (sigma = {sigma_mm:.4f} mm)")

mni_files = sorted(md_mni_dir.glob("*_MD.nii.gz"))
print(f"Found {len(mni_files)} MD maps in MNI to smooth")

Smoothing MD at 5.0 mm FWHM (sigma = 2.1233 mm)
Found 801 MD maps in MNI to smooth


In [82]:
def _smooth_md(md_mni_path):
    dti_stem = md_mni_path.name.replace("_MD.nii.gz", "")
    out_path = md_smooth_dir / f"{dti_stem}_MD_s5.nii.gz"
    if out_path.exists():
        return dti_stem, "skip", ""
    try:
        img = ants.image_read(str(md_mni_path))
        smoothed = ants.smooth_image(img, sigma=sigma_mm, sigma_in_physical_coordinates=True)
        # Scale mm2/s -> 10^-3 mm2/s (WM~0.7, GM~1.0, CSF~3.0)
        ants.image_write(smoothed * 1000.0, str(out_path))
        return dti_stem, "ok", ""
    except Exception as e:
        return dti_stem, "fail", str(e)

In [83]:
failed = []
skipped = 0
ok = 0

max_workers = max(1, (os.cpu_count() or 4) - 2)
print(f"Smoothing with {max_workers} workers")

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(_smooth_md, p) for p in mni_files]
    for fut in tqdm(as_completed(futures), total=len(mni_files), desc="Smooth MD"):
        stem, status, err = fut.result()
        if status == "ok":
            ok += 1
        elif status == "skip":
            skipped += 1
        else:
            failed.append((stem, err))

smoothed_files = list(md_smooth_dir.glob("*_MD_s5.nii.gz"))
print(f"\nMD smoothing complete:")
print(f"  {len(smoothed_files)} smoothed MD maps")
print(f"  ({ok} new, {skipped} skipped, {len(failed)} failed)")
if failed:
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Smoothing with 14 workers


Smooth MD: 100%|██████████| 801/801 [01:02<00:00, 12.75it/s]


MD smoothing complete:
  801 smoothed MD maps
  (801 new, 0 skipped, 0 failed)


#### Fix partial-coverage DTI registrations

119 subjects have limited axial DTI FOV (typically 156–196 mm vs ~255 mm T1).
`BOLDRigid` ignores NIfTI header position and misplaces the partial volume.
Fix: re-register with `Rigid` + `initial_transform='identity'` so ANTs starts
from the header-encoded world position, then cascade the correction through
warp → smooth → parquets.

In [70]:
import nibabel as nib
import numpy as np
import pandas as pd
import shutil
from pathlib import Path

reg_t1_dir   = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm/dti_long_reg_t1_native")
mni_dir      = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm/dti_long_mni")
smooth_dir   = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm/dti_long_mni_smoothed")

# Identify subjects whose registered FA does not reach >= 65% of the T1 z-axis
# or whose z-span is < 50% (catches both bottom-only and top-only partial volumes)
bad_stems = []
for fa_path in sorted(reg_t1_dir.glob("*_FA.nii.gz")):
    stem = fa_path.name.replace("_FA.nii.gz", "")
    fa = nib.load(str(fa_path)).get_fdata()
    nz = np.where(fa > 0.01)
    if len(nz[0]) == 0:
        bad_stems.append(stem)
        continue
    z_max_frac = nz[2].max() / (fa.shape[2] - 1)
    z_span     = (nz[2].max() - nz[2].min()) / (fa.shape[2] - 1)
    if z_max_frac < 0.65 or z_span < 0.50:
        bad_stems.append(stem)

print(f"Subjects needing re-registration: {len(bad_stems)}")

# Delete stale outputs so downstream skip-logic re-runs them
for stem in bad_stems:
    for path in [
        reg_t1_dir / f"{stem}_FA.nii.gz",
        reg_t1_dir / f"{stem}_MD.nii.gz",
        reg_t1_dir / f"{stem}_rigid.mat",
        mni_dir    / f"{stem}_MD.nii.gz",
        smooth_dir / f"{stem}_MD_s5.nii.gz",
    ]:
        if path.exists():
            path.unlink()

print(f"Stale outputs removed — ready to re-run registration")

Subjects needing re-registration: 0
Stale outputs removed — ready to re-run registration


In [71]:
import ants, shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm import tqdm

dtifit_dir   = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm/dti_long_ditfit")
t1_strip_dir = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm/t1_long_skull_strip")
reg_t1_dir   = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm/dti_long_reg_t1_native")

# dti_to_t1 must be in scope from the earlier pairing cell (178f6b51)
# If kernel was restarted rebuild it from paired_df_long.csv
if "dti_to_t1" not in dir():
    import pandas as pd, re as _re
    _paired = pd.read_csv("model_data/adni/paired_df_long.csv")
    dti_to_t1 = dict(zip(_paired["dti_image_subject_id"], _paired["t1_image_subject_id"]))

def _rigid_md_to_t1_fixed(dti_stem: str):
    fa_path = dtifit_dir / dti_stem / f"{dti_stem}_FA.nii.gz"
    md_path = dtifit_dir / dti_stem / f"{dti_stem}_MD.nii.gz"
    t1_stem = dti_to_t1.get(dti_stem)

    out_fa  = reg_t1_dir / f"{dti_stem}_FA.nii.gz"
    out_md  = reg_t1_dir / f"{dti_stem}_MD.nii.gz"
    out_tx  = reg_t1_dir / f"{dti_stem}_rigid.mat"

    if out_md.exists() and out_tx.exists():
        return dti_stem, "skip", ""

    if t1_stem is None:
        return dti_stem, "fail", "no T1 mapping"
    t1_path = t1_strip_dir / f"{t1_stem}.nii.gz"
    if not t1_path.exists():
        return dti_stem, "fail", f"missing T1: {t1_path.name}"
    for p in [fa_path, md_path]:
        if not p.exists():
            return dti_stem, "fail", f"missing {p.name}"

    try:
        fixed     = ants.image_read(str(t1_path))
        moving_fa = ants.image_read(str(fa_path))

        # identity init: ANTs starts from the NIfTI header world-space position
        # instead of centre-of-mass matching, which misfires on partial-FOV DTI
        reg = ants.registration(
            fixed=fixed,
            moving=moving_fa,
            type_of_transform="Rigid",
            initial_transform="identity",
            aff_metric="mattes",
        )

        ants.image_write(reg["warpedmovout"], str(out_fa))

        tx_saved = False
        for tx in reg["fwdtransforms"]:
            tx_p = Path(tx)
            if tx_p.suffix == ".mat" or "GenericAffine" in tx_p.name:
                shutil.copy(tx, out_tx)
                tx_saved = True
                break
        if not tx_saved:
            return dti_stem, "fail", "no .mat in fwdtransforms"

        md_img   = ants.image_read(str(md_path))
        md_in_t1 = ants.apply_transforms(
            fixed=fixed, moving=md_img,
            transformlist=reg["fwdtransforms"],
            interpolator="linear",
        )
        ants.image_write(md_in_t1, str(out_md))
        return dti_stem, "ok", ""

    except Exception as e:
        return dti_stem, "fail", str(e)


failed, ok_count, skipped = [], 0, 0
max_workers = max(1, (os.cpu_count() or 4) - 2)
print(f"Re-registering {len(bad_stems)} partial-coverage subjects ({max_workers} workers)")

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(_rigid_md_to_t1_fixed, s) for s in bad_stems]
    for fut in tqdm(as_completed(futures), total=len(bad_stems), desc="Rigid DTI->T1 (fixed)"):
        stem, status, msg = fut.result()
        if status == "ok":
            ok_count += 1
        elif status == "skip":
            skipped += 1
        else:
            failed.append((stem, msg))

print(f"\nDone: {ok_count} re-registered, {skipped} skipped, {len(failed)} failed")
if failed:
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Re-registering 0 partial-coverage subjects (14 workers)


Rigid DTI->T1 (fixed): 0it [00:00, ?it/s]


Done: 0 re-registered, 0 skipped, 0 failed


In [67]:
import ants
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm import tqdm

t1_warps_dir = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm/t1_long_vbm_warps")
reg_t1_dir   = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm/dti_long_reg_t1_native")
mni_dir      = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm/dti_long_mni")
smooth_dir   = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm/dti_long_mni_smoothed")

mni_t1 = ants.image_read("model_data/mni_t1.nii")

fwhm_mm  = 5.0
sigma_mm = fwhm_mm / (2.0 * np.sqrt(2.0 * np.log(2.0)))

def _rewarp_and_smooth(dti_stem: str):
    t1_stem = dti_to_t1.get(dti_stem)
    if t1_stem is None:
        return dti_stem, "fail", "no T1 mapping"

    md_t1_path = reg_t1_dir / f"{dti_stem}_MD.nii.gz"
    out_mni    = mni_dir    / f"{dti_stem}_MD.nii.gz"
    out_smooth = smooth_dir / f"{dti_stem}_MD_s5.nii.gz"

    if not md_t1_path.exists():
        return dti_stem, "fail", "missing MD in T1 space"

    affine_path = t1_warps_dir / f"{t1_stem}_0GenericAffine.mat"
    warp_path   = t1_warps_dir / f"{t1_stem}_1Warp.nii.gz"
    for p in [affine_path, warp_path]:
        if not p.exists():
            return dti_stem, "fail", f"missing T1 warp: {p.name}"

    try:
        moving = ants.image_read(str(md_t1_path))
        warped = ants.apply_transforms(
            fixed=mni_t1, moving=moving,
            transformlist=[str(warp_path), str(affine_path)],
            interpolator="linear",
        )
        ants.image_write(warped, str(out_mni))

        smoothed = ants.smooth_image(warped, sigma=sigma_mm, sigma_in_physical_coordinates=True)
        # Scale mm2/s -> 10^-3 mm2/s (WM~0.7, GM~1.0, CSF~3.0)
        ants.image_write(smoothed * 1000.0, str(out_smooth))

        return dti_stem, "ok", ""
    except Exception as e:
        return dti_stem, "fail", str(e)


failed, ok_count = [], 0
max_workers = max(1, (os.cpu_count() or 4) - 2)
print(f"Re-warping and re-smoothing {len(bad_stems)} subjects ({max_workers} workers)")

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(_rewarp_and_smooth, s) for s in bad_stems]
    for fut in tqdm(as_completed(futures), total=len(bad_stems), desc="Warp+smooth (fixed)"):
        stem, status, msg = fut.result()
        if status == "ok":
            ok_count += 1
        else:
            failed.append((stem, msg))

print(f"\nDone: {ok_count} warped+smoothed, {len(failed)} failed")
if failed:
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Re-warping and re-smoothing 119 subjects (14 workers)


Warp+smooth (fixed): 100%|██████████| 119/119 [00:30<00:00,  3.93it/s]


Done: 119 warped+smoothed, 0 failed


In [ ]:
import nibabel as nib
import numpy as np
import pandas as pd
import nilearn.image as nli
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from tqdm import tqdm

dti_smooth_dir = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm/dti_long_mni_smoothed")
out_dti_dir    = Path("model_data/adni/dti_long_data")
paired         = pd.read_csv("model_data/adni/paired_df_long.csv")

# Rebuild masks on the current reference grid (in case it changed)
ref_img  = nib.load(str(sorted(dti_smooth_dir.glob("*_MD_s5.nii.gz"))[0]))

def _load_mask(path):
    m = nli.resample_to_img(str(path), ref_img, interpolation="nearest",
                            force_resample=True, copy_header=True)
    return np.asarray(m.dataobj) > 0.5

gm_mask  = _load_mask("model_data/mni_gm_mask.nii")
wm_mask  = _load_mask("model_data/mni_wm_mask.nii")
csf_mask = _load_mask("model_data/mni_csf_mask.nii")

def _extract_md(md_path):
    stem = md_path.name.replace("_MD_s5.nii.gz", "")
    md   = nib.load(str(md_path)).get_fdata(dtype=np.float32)
    return stem, md[gm_mask], md[wm_mask], md[csf_mask]

paired_dti_files = [dti_smooth_dir / f"{s}_MD_s5.nii.gz"
                    for s in paired["dti_image_subject_id"]]
print(f"Rebuilding DTI parquets for {len(paired_dti_files)} subjects")

stems, gm_rows, wm_rows, csf_rows = [], [], [], []
with ThreadPoolExecutor(max_workers=max(1, (os.cpu_count() or 4) - 2)) as ex:
    for stem, gm_vec, wm_vec, csf_vec in tqdm(
        ex.map(_extract_md, paired_dti_files),
        total=len(paired_dti_files), desc="DTI mask -> parquet"
    ):
        stems.append(stem)
        gm_rows.append(gm_vec)
        wm_rows.append(wm_vec)
        csf_rows.append(csf_vec)

dti_gm  = pd.DataFrame(np.vstack(gm_rows),  index=pd.Index(stems, name="image_subject_id"))
dti_wm  = pd.DataFrame(np.vstack(wm_rows),  index=pd.Index(stems, name="image_subject_id"))
dti_csf = pd.DataFrame(np.vstack(csf_rows), index=pd.Index(stems, name="image_subject_id"))
dti_gm.columns  = dti_gm.columns.astype(str)
dti_wm.columns  = dti_wm.columns.astype(str)
dti_csf.columns = dti_csf.columns.astype(str)

dti_gm.to_parquet( out_dti_dir / "dti_long_masked_gm_md.parquet",  compression="zstd")
dti_wm.to_parquet( out_dti_dir / "dti_long_masked_wm_md.parquet",  compression="zstd")
dti_csf.to_parquet(out_dti_dir / "dti_long_masked_csf_md.parquet", compression="zstd")
print(f"Saved dti_long_masked_gm_md.parquet  {dti_gm.shape}")
print(f"Saved dti_long_masked_wm_md.parquet  {dti_wm.shape}")
print(f"Saved dti_long_masked_csf_md.parquet {dti_csf.shape}")

#### ML dataset pre processing

In [85]:
# --- Build GM / WM masks resampled to the smoothed-image grid ----------------
# MNI tissue masks are 1 mm; smoothed PVEs are ~1.5 mm -> resample (NN).
# Use a T1 smoothed image as reference so this cell runs without DTI on disk.
import nilearn.image as nli

t1_smooth_dir = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm/t1_long_pve_mni_smoothed")
dti_smooth_dir = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm/dti_long_mni_smoothed")
out_t1_dir = Path("model_data/adni/t1_long_data")
out_dti_dir = Path("model_data/adni/dti_long_data")
meta_csv = Path("model_data/adni/paired_df_long.csv")

# Prefer a T1 PVE file as reference; fall back to DTI if T1 not yet built
_t1_pve_candidates = sorted(t1_smooth_dir.glob("*/*_pve_1_mod_s2p5.nii.gz"))
_dti_candidates    = sorted(dti_smooth_dir.glob("*_MD_s5.nii.gz"))
if _t1_pve_candidates:
    ref_img = nib.load(str(_t1_pve_candidates[0]))
elif _dti_candidates:
    ref_img = nib.load(str(_dti_candidates[0]))
else:
    raise FileNotFoundError("No reference image found in T1 or DTI smooth dirs")

ref_source = "T1 PVE" if _t1_pve_candidates else "DTI MD"
ref_shape = ref_img.shape

def _load_mask(path):
    m = nli.resample_to_img(str(path), ref_img, interpolation="nearest", force_resample=True, copy_header=True)
    return np.asarray(m.dataobj) > 0.5

gm_mask  = _load_mask("model_data/mni_gm_mask_fast.nii")
wm_mask  = _load_mask("model_data/mni_wm_mask_fast.nii")
csf_mask = _load_mask("model_data/mni_csf_mask_fast.nii")
print(f"Reference grid ({ref_source}): {ref_shape}  |  GM voxels: {gm_mask.sum():,}  |  WM voxels: {wm_mask.sum():,}  |  CSF voxels: {csf_mask.sum():,}")

Reference grid (T1 PVE): (121, 145, 121)  |  GM voxels: 246,466  |  WM voxels: 232,682  |  CSF voxels: 100,004


In [87]:
# --- Build T1 parquets for the paired set ------------------------------------
# Load paired_stems from CSV if available (no DTI files needed on disk).
# Falls back to scanning DTI smooth dir if CSV doesn't exist yet.
from concurrent.futures import ThreadPoolExecutor

def _parse(stem):
    img, _, subj = stem.partition("_")
    return img.lstrip("I"), subj

if meta_csv.exists():
    _paired_csv = pd.read_csv(meta_csv)
    paired_stems = _paired_csv[["t1_image_subject_id", "dti_image_subject_id",
                                "t1_image_id", "dti_image_id", "subject_id"]]
    print(f"Loaded paired_stems from {meta_csv} ({len(paired_stems)} rows)")
else:
    t1_subjects = sorted([p for p in t1_smooth_dir.iterdir() if p.is_dir()])
    dti_files   = sorted(dti_smooth_dir.glob("*_MD_s5.nii.gz"))
    t1_all  = pd.DataFrame([(p.name, *_parse(p.name)) for p in t1_subjects],
                           columns=["t1_image_subject_id", "t1_image_id", "subject_id"])
    dti_all = pd.DataFrame([(s := f.name.replace("_MD_s5.nii.gz", ""), *_parse(s)) for f in dti_files],
                           columns=["dti_image_subject_id", "dti_image_id", "subject_id"])
    t1_all  = t1_all.sort_values(["subject_id", "t1_image_id"]).reset_index(drop=True)
    dti_all = dti_all.sort_values(["subject_id", "dti_image_id"]).reset_index(drop=True)
    t1_all["scan_n"]  = t1_all.groupby("subject_id").cumcount()
    dti_all["scan_n"] = dti_all.groupby("subject_id").cumcount()
    paired_stems = t1_all.merge(dti_all, on=["subject_id", "scan_n"], how="inner").drop(columns="scan_n")
    print(f"T1: {len(t1_all)}  DTI: {len(dti_all)}  Paired: {len(paired_stems)}")

# Filter to subjects whose smoothed PVE files are actually on disk
paired_t1_dirs = [
    d for s in paired_stems["t1_image_subject_id"]
    if (d := t1_smooth_dir / s) and (d / f"{s}_pve_1_mod_s2p5.nii.gz").exists()
]
n_missing = len(paired_stems) - len(paired_t1_dirs)
if n_missing:
    print(f"WARNING: {n_missing} paired subjects missing smoothed T1 files — skipped")
print(f"Extracting T1 parquets for {len(paired_t1_dirs)} subjects")

def _extract_t1(subj_dir):
    stem = subj_dir.name
    csf = nib.load(subj_dir / f"{stem}_pve_0_mod_s2p5.nii.gz").get_fdata(dtype=np.float32)
    gm  = nib.load(subj_dir / f"{stem}_pve_1_mod_s2p5.nii.gz").get_fdata(dtype=np.float32)
    wm  = nib.load(subj_dir / f"{stem}_pve_2_mod_s2p5.nii.gz").get_fdata(dtype=np.float32)
    return stem, gm[gm_mask], wm[wm_mask], csf[csf_mask]

stems, gm_rows, wm_rows, csf_rows = [], [], [], []
with ThreadPoolExecutor(max_workers=max(1, (os.cpu_count() or 4) - 2)) as ex:
    for stem, gm_vec, wm_vec, csf_vec in tqdm(
        ex.map(_extract_t1, paired_t1_dirs), total=len(paired_t1_dirs), desc="T1 mask -> parquet"
    ):
        stems.append(stem); gm_rows.append(gm_vec); wm_rows.append(wm_vec); csf_rows.append(csf_vec)

gm_df  = pd.DataFrame(np.vstack(gm_rows),  index=pd.Index(stems, name="image_subject_id"))
wm_df  = pd.DataFrame(np.vstack(wm_rows),  index=pd.Index(stems, name="image_subject_id"))
csf_df = pd.DataFrame(np.vstack(csf_rows), index=pd.Index(stems, name="image_subject_id"))
gm_df.columns  = gm_df.columns.astype(str)
wm_df.columns  = wm_df.columns.astype(str)
csf_df.columns = csf_df.columns.astype(str)
gm_df.to_parquet(out_t1_dir  / "t1_long_masked_gm.parquet",  compression="zstd")
wm_df.to_parquet(out_t1_dir  / "t1_long_masked_wm.parquet",  compression="zstd")
csf_df.to_parquet(out_t1_dir / "t1_long_masked_csf.parquet", compression="zstd")
print(f"Saved t1_long_masked_gm.parquet {gm_df.shape}, t1_long_masked_wm.parquet {wm_df.shape}, t1_long_masked_csf.parquet {csf_df.shape}")

Loaded paired_stems from model_data/adni/paired_df_long.csv (787 rows)
Extracting T1 parquets for 784 subjects


T1 mask -> parquet: 100%|██████████| 784/784 [00:12<00:00, 61.88it/s] 


Saved t1_long_masked_gm.parquet (784, 246466), t1_long_masked_wm.parquet (784, 232682), t1_long_masked_csf.parquet (784, 100004)


In [6]:
# --- DTI MD -> masked GM-MD / WM-MD parquets (paired set only) ---------------
def _extract_md(md_path):
    stem = md_path.name.replace("_MD_s5.nii.gz", "")
    md = nib.load(md_path).get_fdata(dtype=np.float32)
    return stem, md[gm_mask], md[wm_mask], md[csf_mask]

paired_dti_files = [dti_smooth_dir / f"{s}_MD_s5.nii.gz" for s in paired_stems["dti_image_subject_id"]]
print(f"Building DTI parquets for {len(paired_dti_files)} paired subjects")

stems, gm_rows, wm_rows, csf_rows = [], [], [], []
with ThreadPoolExecutor(max_workers=max(1, (os.cpu_count() or 4) - 2)) as ex:
    for stem, gm_vec, wm_vec, csf_vec in tqdm(ex.map(_extract_md, paired_dti_files), total=len(paired_dti_files), desc="DTI mask -> parquet"):
        stems.append(stem); gm_rows.append(gm_vec); wm_rows.append(wm_vec); csf_rows.append(csf_vec)

dti_gm  = pd.DataFrame(np.vstack(gm_rows),  index=pd.Index(stems, name="image_subject_id"))
dti_wm  = pd.DataFrame(np.vstack(wm_rows),  index=pd.Index(stems, name="image_subject_id"))
dti_csf = pd.DataFrame(np.vstack(csf_rows), index=pd.Index(stems, name="image_subject_id"))
dti_gm.columns = dti_gm.columns.astype(str); dti_wm.columns = dti_wm.columns.astype(str); dti_csf.columns = dti_csf.columns.astype(str)
dti_gm.to_parquet(out_dti_dir  / "dti_long_masked_gm_md.parquet",  compression="zstd")
dti_wm.to_parquet(out_dti_dir  / "dti_long_masked_wm_md.parquet",  compression="zstd")
dti_csf.to_parquet(out_dti_dir / "dti_long_masked_csf_md.parquet", compression="zstd")
print(f"Saved dti_long_masked_gm_md.parquet {dti_gm.shape}, dti_long_masked_wm_md.parquet {dti_wm.shape}, dti_long_masked_csf_md.parquet {dti_csf.shape}")

Building DTI parquets for 787 paired subjects


DTI mask -> parquet: 100%|██████████| 787/787 [00:05<00:00, 150.30it/s]


Saved dti_long_masked_gm_md.parquet (787, 451681), dti_long_masked_wm_md.parquet (787, 283320), dti_long_masked_csf_md.parquet (787, 173822)


In [3]:
# --- paired_df_long.csv : metadata linking T1 + DTI rows to ADNI labels ------
# If kernel was restarted / this cell ran alone: rebuild pairing from disk (no parquet I/O).
import re

meta_csv = globals().get("meta_csv") or Path("model_data/adni/paired_df_long.csv")

paired_stems = globals().get("paired_stems")
if paired_stems is None:
    _t1_sd = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm/t1_long_pve_mni_smoothed")
    _dti_sd = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm/dti_long_mni_smoothed")

    def _stem_parse(stem):
        img, _, subj = stem.partition("_")
        return img.lstrip("I"), subj

    _t1_subj = sorted([p for p in _t1_sd.iterdir() if p.is_dir()])
    _dti_files = sorted(_dti_sd.glob("*_MD_s5.nii.gz"))
    _t1_all = pd.DataFrame([(p.name, *_stem_parse(p.name)) for p in _t1_subj],
                           columns=["t1_image_subject_id", "t1_image_id", "subject_id"])
    _dti_all = pd.DataFrame([(s := f.name.replace("_MD_s5.nii.gz", ""), *_stem_parse(s)) for f in _dti_files],
                            columns=["dti_image_subject_id", "dti_image_id", "subject_id"])
    _t1_all = _t1_all.sort_values(["subject_id", "t1_image_id"]).reset_index(drop=True)
    _dti_all = _dti_all.sort_values(["subject_id", "dti_image_id"]).reset_index(drop=True)
    _t1_all["scan_n"] = _t1_all.groupby("subject_id").cumcount()
    _dti_all["scan_n"] = _dti_all.groupby("subject_id").cumcount()
    paired_stems = _t1_all.merge(_dti_all, on=["subject_id", "scan_n"], how="inner").drop(columns="scan_n")

paired = paired_stems.copy()

# --- Build image_id -> scan_date from DCM folder structure -------------------
# Structure: t1_long_dcm/{subject_id}/{modality}/{YYYY-MM-DD_...}/{image_id}/
_dcm_root = Path("model_data/adni/t1_long_data/t1_long_dcm")
_date_pat = re.compile(r"^(\d{4}-\d{2}-\d{2})")
_img_to_scandate = {}
for _subj_dir in _dcm_root.iterdir():
    if not _subj_dir.is_dir() or _subj_dir.name.startswith("."): continue
    for _mod_dir in _subj_dir.iterdir():
        if not _mod_dir.is_dir() or _mod_dir.name.startswith("."): continue
        for _date_dir in _mod_dir.iterdir():
            if not _date_dir.is_dir() or _date_dir.name.startswith("."): continue
            _m = _date_pat.match(_date_dir.name)
            if not _m: continue
            _scan_date = pd.Timestamp(_m.group(1))
            for _img_dir in _date_dir.iterdir():
                if not _img_dir.is_dir() or _img_dir.name.startswith("."): continue
                _img_to_scandate[_img_dir.name.lstrip("I")] = _scan_date

print(f"Built scan-date map for {len(_img_to_scandate)} T1 image IDs")

# Map each paired row's T1 image_id to its scan date
paired["scan_date"] = pd.to_datetime(paired["t1_image_id"].map(_img_to_scandate))
n_missing = paired["scan_date"].isna().sum()
if n_missing:
    print(f"WARNING: {n_missing} rows could not be matched to a scan date")

# --- DX from DXSUM: closest EXAMDATE to scan date per PTID ------------------
dxsum = pd.read_csv("model_data/adni/DXSUM_22Sep2025.csv",
                    usecols=["PTID", "EXAMDATE", "DIAGNOSIS"], low_memory=False)
dxsum["EXAMDATE"] = pd.to_datetime(dxsum["EXAMDATE"], errors="coerce")
dxsum["DIAGNOSIS"] = pd.to_numeric(dxsum["DIAGNOSIS"], errors="coerce")
dxsum = dxsum.dropna(subset=["EXAMDATE", "DIAGNOSIS"]).loc[lambda z: z["DIAGNOSIS"].isin([1, 2, 3])]

def _closest_dx(row):
    ptid = row["subject_id"]
    scan_dt = row["scan_date"]
    if pd.isna(scan_dt):
        return np.nan
    subset = dxsum[dxsum["PTID"] == ptid]
    if subset.empty:
        return np.nan
    idx = (subset["EXAMDATE"] - scan_dt).abs().idxmin()
    return int(subset.loc[idx, "DIAGNOSIS"])

paired["_dx_int"] = paired.apply(_closest_dx, axis=1)
paired["group"] = paired["_dx_int"].map({1: "CN", 2: "MCI", 3: "Dementia"})
paired["diag_label"] = paired["_dx_int"].map({1: 0, 2: 1, 3: 1})
paired = paired.drop(columns=["_dx_int", "scan_date"])

# --- Amyloid status: most recent non-null per PTID (unchanged) ---------------
amy = pd.read_csv("model_data/adni/All_Subjects_UCBERKELEY_AMY_6MM_08Feb2026.csv",
                  usecols=["PTID", "SCANDATE", "AMYLOID_STATUS"], low_memory=False)
amy = amy.dropna(subset=["AMYLOID_STATUS"]).sort_values("SCANDATE").drop_duplicates("PTID", keep="last")
paired["amyloid_label"] = paired["subject_id"].map(dict(zip(amy["PTID"], amy["AMYLOID_STATUS"]))).astype("Int64")

paired = paired[["subject_id",
                 "t1_image_id", "dti_image_id",
                 "t1_image_subject_id", "dti_image_subject_id",
                 "group", "diag_label", "amyloid_label"]]
paired.to_csv(meta_csv, index=False)
print(f"Saved {meta_csv}  ({len(paired)} rows)")
print(paired["group"].value_counts(dropna=False).to_dict(),
      "| amyloid:", paired["amyloid_label"].value_counts(dropna=False).to_dict())

Built scan-date map for 802 T1 image IDs
Saved model_data/adni/paired_df_long.csv  (787 rows)
{'CN': 421, 'MCI': 286, 'Dementia': 80} | amyloid: {np.int64(1): 376, np.int64(0): 348, <NA>: 63}
